# Increase Resolution — distributed (tile-level) walkthrough

This notebook shows the **tile-level** flow, for spreading the GPU work across **multiple worker machines**. The pipeline splits into three steps so the heavy per-tile inference can run on your own fleet:

1. **`split`** (coordinator, CPU/no engine) — cut the image into independent tiles.
2. **`TileWorker.infer`** (each worker machine, GPU) — upscale the tiles it receives.
3. **`merge`** (coordinator, CPU) — stitch the upscaled tiles back together.

The result is identical to the single-call `execute` (see `code_example.ipynb`) — distributing the work does not change the output.

**Prerequisites:** complete the *Runtime setup* and installation from the main README / `code_example.ipynb` first (base image, env vars, install — this single-machine demo needs `increase-resolution[all]`; in production a coordinator uses `[cpu]` and each worker uses `[gpu]`, and the downloaded `.engine` files). This notebook assumes the package is installed and the engine paths are available.

## 0. Config — engine paths
Point these at the `.engine` files you downloaded (same as the simple notebook).

In [ ]:
import os

CACHE = os.path.expanduser("~/.cache/bria/increase-resolution/")
ENGINE_PATHS = {
    2: os.path.join(CACHE, "increase_resolution2.engine"),
    4: os.path.join(CACHE, "increase_resolution4.engine"),
}
SCALE = 4
SAMPLE = "https://bria-test-images.s3.us-east-1.amazonaws.com/starry_2000x2000.png"

## 1. Split — on the coordinator (no GPU/engine needed)
`split` returns independent tiles (plain numpy arrays), a `layout` for reassembly, and the input alpha (if any). The tiles are what you ship to your workers.

In [ ]:
from bria_external.image import ImageModel
from increase_resolution import split

image = ImageModel(image=SAMPLE).image
res = split(image, scale=SCALE)
print(f"{len(res.tiles)} tiles of shape {res.tiles[0].shape} (send these to workers)")

## 2. Tile inference — on your worker machines (GPU)
This is the **tile-only GPU processing**. Each worker machine loads the engine **once** and runs `infer` on the tiles it receives.

> **Replace the loop below with your cross-machine transport.** In production you ship `res.tiles` to your GPU workers (queue / RPC / Ray / gRPC / ...), each worker runs `TileWorker(engine).infer(tile)`, and you collect the upscaled tiles back **in order**. Here we use one local worker so the notebook is self-contained.

In [ ]:
from increase_resolution import TileWorker

worker = TileWorker(ENGINE_PATHS[SCALE]).setup()   # each worker machine does this once
upscaled = [worker.infer(tile) for tile in res.tiles]   # <-- your fan-out goes here
worker.close()
print(f"upscaled {len(upscaled)} tiles")

## 3. Merge — back on the coordinator
Stitch the upscaled tiles into the final image (overlaps are averaged).

In [ ]:
from increase_resolution import merge
from IPython.display import display

out = merge(upscaled, res.layout, res.alpha)
print("output size:", out.size)
out.save("output_distributed.png")
display(out)

## 4. (Optional) correctness — distributed == single-call
Sanity check: the distributed result matches `IncreaseResolution.execute` exactly.

In [ ]:
import numpy as np
from increase_resolution import IncreaseResolution, IncreaseResolutionConfig, IncreaseResolutionInput

pipeline = IncreaseResolution(config=IncreaseResolutionConfig(engine_paths=ENGINE_PATHS))
pipeline.setup()
ref = pipeline.execute(IncreaseResolutionInput(image=image, desired_increase=SCALE)).image
pipeline.cleanup()

mae = np.abs(np.asarray(out).astype(int) - np.asarray(ref).astype(int)).mean()
print(f"distributed vs single-call MAE = {mae:.4f}/255  ({'MATCH' if mae < 1.0 else 'MISMATCH'})")